# Prepares the apps dataset for experiments

This notebook prepares the WinApps dataset artifacts used by the downstream experiment notebooks.

The suggested temporal split keeps the newest month fully unseen for final evaluation while still reserving the preceding month for validation:

- train: `2024-07-01` to `2025-02-28` (`meta.sample.id >= "20240701-"` and `< "20250301-"`)
- validation: `2025-03-01` to `2025-03-31` (`meta.sample.id >= "20250301-"` and `< "20250401-"`)
- test: `2025-04-01` onward in the current dataset (`meta.sample.id >= "20250401-"` and `< "20250501-"`)

This split preserves temporal ordering, keeps most of the historical data for fitting the encoder and preprocessor, and still leaves separate validation and test windows in the most recent period.

The notebook uses the shared `get_connection_label(...)` helper so the labeling stays consistent with the malware preparation workflow. The saved label parquet files contain the returned `category`, `label`, and `platform` columns. For `winapps.parquet`, the categories naturally reduce to `application`, `system`, and `unknown`.

In [2]:
import sys
sys.path.append('../src')

from pathlib import Path
import joblib
import pyarrow.compute as pc
from tls_profiling.io.readers import open_parquet_dataset

TRAIN_START = "20240701-"
VAL_START = "20250301-"
TEST_START = "20250401-"
END_EXCLUSIVE = "20250501-"

dataset = open_parquet_dataset("../datasets/winapps.parquet")
print("Loading dataset...")
df_train = dataset.to_table(
    filter=((pc.field("meta.sample.id") >= TRAIN_START) & (pc.field("meta.sample.id") < VAL_START))
).to_pandas()
print(f"df_apps_train={len(df_train)}")
df_val = dataset.to_table(
    filter=((pc.field("meta.sample.id") >= VAL_START) & (pc.field("meta.sample.id") < TEST_START))
).to_pandas()
print(f"df_apps_val={len(df_val)}")
df_test = dataset.to_table(
    filter=((pc.field("meta.sample.id") >= TEST_START) & (pc.field("meta.sample.id") < END_EXCLUSIVE))
).to_pandas()
print(f"df_apps_test={len(df_test)}")

from tls_profiling.exploration.connections import get_connection_label
labels_train = get_connection_label(df_train)
labels_val = get_connection_label(df_val)
labels_test = get_connection_label(df_test)

from tls_profiling.preprocessing import extract_features, build_and_fit_preprocessor

print("Extracting features from source data.")
# transform loaded datasets to feature input for models:
ef_train = extract_features(df_train)
print(" Train data done.")
ef_val = extract_features(df_val)
print(" Validation data done.")
ef_test = extract_features(df_test)
print(" Test data done.")

print("Fitting preprocessor.")
# fit preprocessors that learns the categorical features encoding
preprocessor = build_and_fit_preprocessor(ef_train)
print("Preprocessing done.")

artifacts_dir = Path("./data")
artifacts_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving extracted feature splits to {artifacts_dir}.")
ef_train.to_parquet(artifacts_dir / "apps_train.parquet", index=False)
ef_val.to_parquet(artifacts_dir / "apps_val.parquet", index=False)
ef_test.to_parquet(artifacts_dir / "apps_test.parquet", index=False)

print(f"Saving label splits to {artifacts_dir}.")
labels_train.to_parquet(artifacts_dir / "apps_train_labels.parquet", index=False)
labels_val.to_parquet(artifacts_dir / "apps_val_labels.parquet", index=False)
labels_test.to_parquet(artifacts_dir / "apps_test_labels.parquet", index=False)

preprocessor_path = artifacts_dir / "apps_preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)
print(f"Saved preprocessor to {preprocessor_path}.")


Loading dataset...
df_apps_train=499433
df_apps_val=47816
df_apps_test=15629
Extracting features from source data.
tls.sext.=0012
tls.sext.=0029
tls.ccs.=0007
tls.ccs.=000D
tls.ccs.=0010
tls.ccs.=0013
tls.ccs.=0016
tls.ccs.=0030
tls.ccs.=0031
tls.ccs.=0036
tls.ccs.=0037
tls.ccs.=003E
tls.ccs.=003F
tls.ccs.=0040
tls.ccs.=0041
tls.ccs.=0042
tls.ccs.=0043
tls.ccs.=0044
tls.ccs.=0045
tls.ccs.=0067
tls.ccs.=0068
tls.ccs.=0069
tls.ccs.=006A
tls.ccs.=006B
tls.ccs.=0084
tls.ccs.=0085
tls.ccs.=0086
tls.ccs.=0087
tls.ccs.=0088
tls.ccs.=008C
tls.ccs.=008D
tls.ccs.=0090
tls.ccs.=0091
tls.ccs.=0094
tls.ccs.=0095
tls.ccs.=0096
tls.ccs.=0097
tls.ccs.=0098
tls.ccs.=0099
tls.ccs.=009A
tls.ccs.=00A0
tls.ccs.=00A1
tls.ccs.=00A2
tls.ccs.=00A3
tls.ccs.=00A4
tls.ccs.=00A5
tls.ccs.=00A8
tls.ccs.=00A9
tls.ccs.=00AA
tls.ccs.=00AB
tls.ccs.=00AC
tls.ccs.=00AD
tls.ccs.=00AE
tls.ccs.=00AF
tls.ccs.=00B2
tls.ccs.=00B3
tls.ccs.=00B6
tls.ccs.=00B7
tls.ccs.=00BA
tls.ccs.=00BD
tls.ccs.=00BE
tls.ccs.=00C0
tls.ccs.=00C3
t

# Prepare Data using EXT feature extractor

This section repeats the same split, labeling, and preprocessing workflow, but replaces the standard extractor with `extract_features_ext(...)`.

The EXT representation is stored separately in `./data-ext` so it can be compared against the standard feature set without overwriting any artifacts.

In [3]:
import sys
sys.path.append('../src')

from pathlib import Path
import joblib
import pyarrow.compute as pc
from tls_profiling.io.readers import open_parquet_dataset

TRAIN_START = "20240701-"
VAL_START = "20250301-"
TEST_START = "20250401-"
END_EXCLUSIVE = "20250501-"

dataset = open_parquet_dataset("../datasets/winapps.parquet")
print("Loading dataset...")
df_train = dataset.to_table(
    filter=((pc.field("meta.sample.id") >= TRAIN_START) & (pc.field("meta.sample.id") < VAL_START))
).to_pandas()
print(f"df_apps_train={len(df_train)}")
df_val = dataset.to_table(
    filter=((pc.field("meta.sample.id") >= VAL_START) & (pc.field("meta.sample.id") < TEST_START))
).to_pandas()
print(f"df_apps_val={len(df_val)}")
df_test = dataset.to_table(
    filter=((pc.field("meta.sample.id") >= TEST_START) & (pc.field("meta.sample.id") < END_EXCLUSIVE))
).to_pandas()
print(f"df_apps_test={len(df_test)}")

from tls_profiling.exploration.connections import get_connection_label
labels_train = get_connection_label(df_train)
labels_val = get_connection_label(df_val)
labels_test = get_connection_label(df_test)

from tls_profiling.preprocessing import extract_features_ext, build_and_fit_preprocessor

print("Extracting features from source data.")
# transform loaded datasets to feature input for models:
ef_train = extract_features_ext(df_train)
print(" Train data done.")
ef_val = extract_features_ext(df_val)
print(" Validation data done.")
ef_test = extract_features_ext(df_test)
print(" Test data done.")

print("Fitting preprocessor.")
# fit preprocessors that learns the categorical features encoding
preprocessor = build_and_fit_preprocessor(ef_train)
print("Preprocessing done.")

artifacts_dir = Path("./data-ext")
artifacts_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving extracted feature splits to {artifacts_dir}.")
ef_train.to_parquet(artifacts_dir / "apps_train.parquet", index=False)
ef_val.to_parquet(artifacts_dir / "apps_val.parquet", index=False)
ef_test.to_parquet(artifacts_dir / "apps_test.parquet", index=False)

print(f"Saving label splits to {artifacts_dir}.")
labels_train.to_parquet(artifacts_dir / "apps_train_labels.parquet", index=False)
labels_val.to_parquet(artifacts_dir / "apps_val_labels.parquet", index=False)
labels_test.to_parquet(artifacts_dir / "apps_test_labels.parquet", index=False)

preprocessor_path = artifacts_dir / "apps_preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)
print(f"Saved preprocessor to {preprocessor_path}.")


Loading dataset...
df_apps_train=499433
df_apps_val=47816
df_apps_test=15629
Extracting features from source data.
tls.sext.=0012
tls.sext.=0029
tls.ccs.=0007
tls.ccs.=000D
tls.ccs.=0010
tls.ccs.=0013
tls.ccs.=0016
tls.ccs.=0030
tls.ccs.=0031
tls.ccs.=0036
tls.ccs.=0037
tls.ccs.=003E
tls.ccs.=003F
tls.ccs.=0040
tls.ccs.=0041
tls.ccs.=0042
tls.ccs.=0043
tls.ccs.=0044
tls.ccs.=0045
tls.ccs.=0067
tls.ccs.=0068
tls.ccs.=0069
tls.ccs.=006A
tls.ccs.=006B
tls.ccs.=0084
tls.ccs.=0085
tls.ccs.=0086
tls.ccs.=0087
tls.ccs.=0088
tls.ccs.=008C
tls.ccs.=008D
tls.ccs.=0090
tls.ccs.=0091
tls.ccs.=0094
tls.ccs.=0095
tls.ccs.=0096
tls.ccs.=0097
tls.ccs.=0098
tls.ccs.=0099
tls.ccs.=009A
tls.ccs.=00A0
tls.ccs.=00A1
tls.ccs.=00A2
tls.ccs.=00A3
tls.ccs.=00A4
tls.ccs.=00A5
tls.ccs.=00A8
tls.ccs.=00A9
tls.ccs.=00AA
tls.ccs.=00AB
tls.ccs.=00AC
tls.ccs.=00AD
tls.ccs.=00AE
tls.ccs.=00AF
tls.ccs.=00B2
tls.ccs.=00B3
tls.ccs.=00B6
tls.ccs.=00B7
tls.ccs.=00BA
tls.ccs.=00BD
tls.ccs.=00BE
tls.ccs.=00C0
tls.ccs.=00C3
t